# EmbracePlus Raw Data Batch Downloader — user-wise, all modalities

This notebook:
1. Connects to your Empatica S3 bucket using your Access Key.
2. Scans every account (device) from **March 2026 onward**.
3. Matches each day's data to the right **anonymized user folder** (user1, user2, ... — no real names anywhere) using the session table below.
4. Extracts **all 10 raw modalities** (accelerometer, gyroscope, magnetometer, eda, temperature, bvp, systolicPeaks, steps, tags, ambientLight) with correct unit conversions.
5. Saves one CSV per modality, per session, into `output/userN/`, each tagged with account number, date, and both UTC + IST timestamps.

**Before running:** fill in Cell 2 (credentials) and check Cell 4 (org/study/site IDs) and Cell 5 (session table) match your setup. Rows marked `# REVIEW` came from ambiguous entries in your original table — double check them.


In [15]:
# Cell 1 — install dependencies (run once)
%pip install boto3 fastavro pandas --quiet


Note: you may need to restart the kernel to use updated packages.


In [16]:
# Cell 2 — YOUR CREDENTIALS (fill these in)
ACCESS_KEY = "<ACCESS_KEY>"
SECRET_KEY = "<REMOVED>"
REGION     = "us-east-1"

BUCKET = "empatica-us-east-1-prod-data"


In [17]:
# Cell 3 — S3 client setup
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
)
print("S3 client ready.")


S3 client ready.


In [18]:
# Cell 4 — org / study / site path (from your S3 Access URL, e.g. s3://.../v2/2753/1/1/...)
ORG_ID   = "2753"
STUDY_ID = "1"
SITE_ID  = "1"

BASE_PREFIX = f"v2/{ORG_ID}/{STUDY_ID}/{SITE_ID}"
print("Base prefix:", BASE_PREFIX)


Base prefix: v2/2753/1/1


In [19]:
# Cell 5 — SESSION TABLE (anonymized users only — no real names)
# Each row: user folder, account number, start_date, end_date (or None), start_time (or None), end_time (or None), exclude flag
# Dates are 'YYYY-MM-DD'. Times are 'HH:MM' in 24-hour IST. None means "not specified / full day / open end".
# Obvious year typos from the original sheet (2027/2028/2029) have been corrected to 2026 — verify these ('# REVIEW').

SESSION_TABLE = [
    # user, account, start_date,   end_date,      start_time, end_time, exclude
    ("user1",  0, "2026-03-22", None,           "19:05", "20:36", False),
    ("user2",  0, "2026-03-22", None,           "12:40", "13:30", False),
    ("user3",  0, "2026-05-18", None,           "16:08", "00:00", False),
    ("user4",  5, "2026-05-19", "2026-06-03",   "12:20", None,    False),  # multi-day, verify daily presence
    ("user5",  0, "2026-05-22", None,           "19:00", "20:00", False),
    ("user5",  0, "2026-05-28", None,           "20:00", "21:30", True),   # REJECTED — excluded
    ("user6",  0, "2026-05-29", None,           "21:15", "23:04", False),
    ("user7",  0, "2026-05-30", None,           "18:50", "20:50", False),
    ("user8",  0, "2026-05-30", None,           "20:50", "21:15", False),
    ("user9",  0, "2026-05-31", None,           "16:00", "17:24", False),
    ("user10", 0, "2026-05-31", None,           "17:25", "17:55", False),
    ("user11", 0, "2026-05-04", None,           "19:00", "20:22", False),
    ("user12", 0, "2026-05-04", None,           "20:22", "20:40", False),
    ("user14", 5, "2026-06-03", None,           "15:30", None,    False),  # open end -> whole rest of day
    ("user15", 5, "2026-06-10", "2026-06-11",   "21:00", "08:00", False),  # overnight
    ("user16", 6, "2026-06-23", "2026-06-30",   None,    None,    False),  # whole range, account-level
    ("user17", 6, "2026-07-01", "2026-07-09",   None,    None,    False),  # whole range, account-level
    ("user19", 5, "2026-07-16", None,           "13:28", "12:00", False),  # REVIEW: crosses to next day noon?
    ("user1",  0, "2026-07-13", None,           "16:16", "17:17", False),
    ("user3",  0, "2026-07-19", None,           "16:40", "00:00", False),
    ("user14", 0, "2026-07-17", None,           "19:56", "22:20", False),  # REVIEW: same user id, different account (0 vs 5) — confirm this is really user14
    ("user3",  0, "2026-07-20", None,           "13:00", "00:00", False),
    ("user3",  0, "2026-07-21", None,           "12:00", "00:12", False),
    ("user3",  0, "2026-07-22", None,           "10:00", "15:09", False),
    ("user3",  0, "2026-07-22", None,           "16:00", "23:59", False),  # REVIEW: second session same day, end time unclear in original
    ("user3",  0, "2026-07-26", None,           "14:00", "22:00", False),
    ("user18", 2, None,          None,          None,    None,    False),  # ALL of account 2, always, no date/time filter
    ("user2",  0, "2026-07-27", "2026-07-31",   None,    None,    False),  # explicit multi-day claim, verify daily presence
    ("user20", 6, "2026-08-12", "2026-08-13",   None,    None,    False),
    ("user4",  5, "2026-07-15", "2026-07-18",   None,    None,    False),
    ("user4",  5, "2026-07-23", "2026-07-23",   None,    None,    False),
    ("user4",  5, "2026-07-25", None,           None,    None,    False),
    ("user21", 6, "2026-08-14", None,           "15:30", "23:59", False),  # REVIEW: original end time '12.00pm' unclear
    ("user21", 0, "2026-08-15", None,           "11:31", "13:30", False),
]

# Fallback rules applied AFTER checking SESSION_TABLE for an explicit match:
#   account 0, unclaimed by any row above -> user2
#   account 2, always                     -> user18 (already covered above, kept here as a safety net)
FALLBACK_BY_ACCOUNT = {
    0: "user2",
    2: "user18",
}

print(f"{len(SESSION_TABLE)} session rows loaded.")


34 session rows loaded.


In [20]:
# Cell 6 — date helpers
import pandas as pd
from datetime import datetime, timedelta

IST_OFFSET = pd.Timedelta(hours=5, minutes=30)
SCAN_START_DATE = pd.Timestamp("2026-03-01")   # scan from March 2026 onward
SCAN_END_DATE   = pd.Timestamp.today().normalize()

def daterange(start, end):
    days = []
    d = start
    while d <= end:
        days.append(d.strftime("%Y-%m-%d"))
        d += timedelta(days=1)
    return days

def parse_hhmm(date_str, hhmm):
    """Combine a date string and 'HH:MM' (IST) into a UTC Timestamp."""
    if hhmm is None:
        return None
    ist_dt = pd.Timestamp(f"{date_str} {hhmm}:00")
    return ist_dt - IST_OFFSET


In [21]:
# Cell 7 — figure out which account+date windows belong to which user
# Builds a lookup: for a given (account, date), a list of (user, start_utc_or_None, end_utc_or_None, whole_day)

def build_ownership_index():
    idx = {}
    for user, account, sdate, edate, stime, etime, exclude in SESSION_TABLE:
        if exclude:
            continue
        if account == 2:
            # account 2 belongs entirely to this user, no date filtering
            idx.setdefault(("ALL_DATES", account), []).append((user, None, None, True))
            continue
        end_date = edate if edate else sdate
        for date_str in daterange(pd.Timestamp(sdate), pd.Timestamp(end_date)):
            whole_day = (stime is None and etime is None)
            start_utc = parse_hhmm(date_str, stime) if stime else None
            # handle overnight end times (e.g. '00:00' or times earlier than start) by rolling to next day
            end_utc = None
            if etime:
                end_utc = parse_hhmm(date_str, etime)
                if start_utc is not None and end_utc <= start_utc:
                    end_utc += pd.Timedelta(days=1)
            idx.setdefault((date_str, account), []).append((user, start_utc, end_utc, whole_day))
    return idx

OWNERSHIP_INDEX = build_ownership_index()
print(f"Built ownership index for {len(OWNERSHIP_INDEX)} (date, account) keys.")


Built ownership index for 65 (date, account) keys.


In [22]:
# Cell 8 — S3 listing helpers

def list_participant_folders(date_str):
    """List participant subfolders (e.g. '0006-3YK9J1H1C7') under participant_data/{date}/"""
    prefix = f"{BASE_PREFIX}/participant_data/{date_str}/"
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    folders = []
    for cp in resp.get("CommonPrefixes", []):
        name = cp["Prefix"].rstrip("/").split("/")[-1]   # e.g. '0006-3YK9J1H1C7'
        folders.append(name)
    return folders

def account_from_folder(folder_name):
    """'0006-3YK9J1H1C7' -> account number 6"""
    try:
        return int(folder_name.split("-")[0])
    except Exception:
        return None

def list_avro_files(date_str, participant_folder):
    prefix = f"{BASE_PREFIX}/participant_data/{date_str}/{participant_folder}/raw_data/v6/"
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return [obj["Key"] for obj in resp.get("Contents", [])]


In [23]:
# Cell 9 — owner lookup for a given account/date/timestamp

def find_owner(account, date_str, sample_utc_ts):
    # account 2 -> always user18
    if ("ALL_DATES", account) in OWNERSHIP_INDEX:
        return OWNERSHIP_INDEX[("ALL_DATES", account)][0][0]

    candidates = OWNERSHIP_INDEX.get((date_str, account), [])
    for user, start_utc, end_utc, whole_day in candidates:
        if whole_day:
            return user
        if start_utc is not None and end_utc is not None:
            if start_utc <= sample_utc_ts <= end_utc:
                return user
        elif start_utc is not None and end_utc is None:
            if sample_utc_ts >= start_utc:
                return user

    # fallback rules
    fallback_user = FALLBACK_BY_ACCOUNT.get(account)
    if fallback_user:
        return fallback_user

    return f"unassigned_account{account}"


In [24]:
# Cell 10 — extract ALL 10 raw modalities from one avro file, with unit conversion + UTC/IST timestamps

from fastavro import reader as avro_reader
import io

FLAT_SENSORS = ["eda", "temperature", "bvp", "steps", "ambientLight"]
AXIS_SENSORS = ["accelerometer", "gyroscope", "magnetometer"]
TIMESTAMP_ONLY = ["systolicPeaks", "tags"]

def extract_all_modalities(file_bytes, account, date_str):
    """Returns dict: sensor_name -> DataFrame (with account_number, date, datetime_utc, datetime_ist columns)"""
    result = {}
    buf = io.BytesIO(file_bytes)
    for record in avro_reader(buf):
        raw = record["rawData"]

        # --- flat single-value sensors ---
        for sensor in FLAT_SENSORS:
            s = raw.get(sensor)
            if not s or s.get("samplingFrequency", 0) == 0:
                continue
            fs = s["samplingFrequency"]
            start_ts = s["timestampStart"]
            n = len(s["values"])
            ts_utc = pd.to_datetime([start_ts + i * (1e6 / fs) for i in range(n)], unit="us")
            df = pd.DataFrame({"datetime_utc": ts_utc, sensor: s["values"]})
            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result[sensor] = df

        # --- 3-axis sensors ---
        for sensor in AXIS_SENSORS:
            s = raw.get(sensor)
            if not s or s.get("samplingFrequency", 0) == 0:
                continue
            fs = s["samplingFrequency"]
            start_ts = s["timestampStart"]
            n = len(s["x"])
            ts_utc = pd.to_datetime([start_ts + i * (1e6 / fs) for i in range(n)], unit="us")

            if sensor in ("accelerometer", "gyroscope"):
                params = s["imuParams"]
                cf = params.get("conversionFactor")
                if cf:
                    x = [v * cf for v in s["x"]]
                    y = [v * cf for v in s["y"]]
                    z = [v * cf for v in s["z"]]
                else:
                    dphys = params["physicalMax"] - params["physicalMin"]
                    ddig = params["digitalMax"] - params["digitalMin"]
                    x = [v * dphys / ddig for v in s["x"]]
                    y = [v * dphys / ddig for v in s["y"]]
                    z = [v * dphys / ddig for v in s["z"]]
                df = pd.DataFrame({
                    "datetime_utc": ts_utc,
                    f"{sensor}_x": x, f"{sensor}_y": y, f"{sensor}_z": z,
                })
            else:  # magnetometer — already physical units, has quality index
                df = pd.DataFrame({
                    "datetime_utc": ts_utc,
                    "magnetometer_x": s["x"], "magnetometer_y": s["y"], "magnetometer_z": s["z"],
                    "quality_index": s.get("qualityIndex", [None]*n),
                })

            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result[sensor] = df

        # --- timestamp-only sensors ---
        sp = raw.get("systolicPeaks")
        if sp and sp.get("peaksTimeNanos"):
            ts_utc = pd.to_datetime(sp["peaksTimeNanos"], unit="ns")
            df = pd.DataFrame({"datetime_utc": ts_utc})
            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result["systolicPeaks"] = df

        tags = raw.get("tags")
        if tags and tags.get("tagsTimeMicros"):
            ts_utc = pd.to_datetime(tags["tagsTimeMicros"], unit="us")
            df = pd.DataFrame({"datetime_utc": ts_utc})
            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result["tags"] = df

        break  # one record per avro file
    return result


In [25]:
# Cell 11 — split extracted sensor data by owner (a file/date can belong to multiple users if times split within a day)

def split_by_owner(sensor_dfs, account, date_str):
    """Returns dict: user -> {sensor_name: DataFrame subset for that user}"""
    out = {}
    for sensor, df in sensor_dfs.items():
        if df.empty:
            continue
        owners = df["datetime_utc"].apply(lambda ts: find_owner(account, date_str, ts))
        for user in owners.unique():
            sub = df[owners == user]
            out.setdefault(user, {})[sensor] = sub
    return out


In [26]:
# Cell 12 — save helper: append per-user, per-sensor CSVs to output/userN/

import os

OUTPUT_DIR = "output"

def save_user_sensor_data(user, account, date_str, sensor, df):
    user_dir = os.path.join(OUTPUT_DIR, user)
    os.makedirs(user_dir, exist_ok=True)
    fname = f"{user}_acct{account}_{date_str}_{sensor}.csv"
    fpath = os.path.join(user_dir, fname)
    write_header = not os.path.exists(fpath)
    df.to_csv(fpath, mode="a", index=False, header=write_header)


In [27]:
# Cell 12b — resume / skip-already-processed manifest

MANIFEST_PATH = os.path.join(OUTPUT_DIR, "processed_files.log")

def load_manifest():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if not os.path.exists(MANIFEST_PATH):
        return set()
    with open(MANIFEST_PATH, "r") as f:
        return set(line.strip() for line in f if line.strip())

def mark_processed(key):
    # append immediately so a crash/network drop never loses progress
    with open(MANIFEST_PATH, "a") as f:
        f.write(key + "\n")

PROCESSED_KEYS = load_manifest()
print(f"{len(PROCESSED_KEYS)} files already processed in previous runs — these will be skipped.")


0 files already processed in previous runs — these will be skipped.


In [28]:
# Cell 13 — MAIN LOOP (resumable): scan every date from March 2026 onward, skip already-processed files, download, extract, assign, save

import time
from botocore.exceptions import EndpointConnectionError, ClientError

MAX_RETRIES = 5
RETRY_WAIT_SECONDS = 5

all_dates = daterange(SCAN_START_DATE, SCAN_END_DATE)
processed_files = 0
skipped_already_done = 0
skipped_dates = 0

for date_str in all_dates:
    folders = list_participant_folders(date_str)
    if not folders:
        skipped_dates += 1
        continue

    for folder in folders:
        account = account_from_folder(folder)
        if account is None:
            continue

        avro_keys = list_avro_files(date_str, folder)
        if not avro_keys:
            continue

        for key in avro_keys:
            if key in PROCESSED_KEYS:
                skipped_already_done += 1
                continue

            # retry loop for transient network errors
            file_bytes = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    obj = s3.get_object(Bucket=BUCKET, Key=key)
                    file_bytes = obj["Body"].read()
                    break
                except (EndpointConnectionError, ClientError) as e:
                    print(f"  [retry {attempt}/{MAX_RETRIES}] network issue on {key}: {e}")
                    time.sleep(RETRY_WAIT_SECONDS)
            else:
                print(f"  [SKIPPED after {MAX_RETRIES} retries] {key}")
                continue  # give up on this file, move to next, but do NOT mark as processed

            sensor_dfs = extract_all_modalities(file_bytes, account, date_str)
            if not sensor_dfs:
                mark_processed(key)
                PROCESSED_KEYS.add(key)
                continue

            owner_split = split_by_owner(sensor_dfs, account, date_str)

            for user, sensors in owner_split.items():
                for sensor, df in sensors.items():
                    save_user_sensor_data(user, account, date_str, sensor, df)

            mark_processed(key)
            PROCESSED_KEYS.add(key)
            processed_files += 1

            if processed_files % 20 == 0:
                print(f"Processed {processed_files} new files so far... (currently on {date_str})")

print(f"\nDone. Processed {processed_files} new avro files. "
      f"Skipped {skipped_already_done} already-done files. "
      f"{skipped_dates} dates had no data at all.")
print("You can safely re-run this cell any time — it will pick up where it left off.")


Processed 20 new files so far... (currently on 2026-03-11)
Processed 40 new files so far... (currently on 2026-03-15)
Processed 60 new files so far... (currently on 2026-03-15)
Processed 80 new files so far... (currently on 2026-03-15)
Processed 100 new files so far... (currently on 2026-03-16)
Processed 120 new files so far... (currently on 2026-03-16)
Processed 140 new files so far... (currently on 2026-03-17)
Processed 160 new files so far... (currently on 2026-03-17)
Processed 180 new files so far... (currently on 2026-03-17)
Processed 200 new files so far... (currently on 2026-03-18)
Processed 220 new files so far... (currently on 2026-03-19)
Processed 240 new files so far... (currently on 2026-03-21)
Processed 260 new files so far... (currently on 2026-03-21)
Processed 280 new files so far... (currently on 2026-03-22)
Processed 300 new files so far... (currently on 2026-03-22)
Processed 320 new files so far... (currently on 2026-04-13)
Processed 340 new files so far... (currently

## Notes / things to double-check after running

- Rows marked `# REVIEW` in Cell 5 came from ambiguous entries in your original table (crossing-midnight times written inconsistently, a possible duplicate account for `user14`, an unclear second session for `user3` on 22/07). Verify these against your original records.
- Any account/date combination not covered by an explicit row and not account 0 or 2 will be saved under `unassigned_accountN` — check the `output/` folder for any such folders and assign them manually if needed.
- All output filenames follow the pattern `userN_acctK_YYYY-MM-DD_sensorname.csv` — no real names anywhere, only anonymized user IDs, account numbers, and dates.
- Every row inside every CSV carries `account_number`, `date`, `datetime_utc`, and `datetime_ist` columns for full traceability.
